# Part A Question 4: Bigram Sentence Probabilities

This notebook computes the target sentence probability using unsmoothed and add-one smoothed bigram language models.

In [1]:
import re
from collections import Counter
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import pandas as pd

In [2]:
DATA_PATH = Path('data/Data_3.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_3.txt')

text = DATA_PATH.read_text(encoding='utf-8')
print(text)

Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a different book by Danielle </s>



## 1. Pre-processing

In [3]:
def extract_padded_sentences(text: str) -> list[list[str]]:
    """Extract all <s> ... </s> sentences from the question text."""

    matches = re.findall(r'<s>\s+(.*?)\s+</s>', text)
    return [['<s>', *match.split(), '</s>'] for match in matches]


sentences = extract_padded_sentences(text)
training_sentences = sentences[:-1]
target_sentence = sentences[-1]

print('Training sentences:')
for sentence in training_sentences:
    print(sentence)

print('\nTarget sentence:')
print(target_sentence)

Training sentences:
['<s>', 'He', 'read', 'a', 'book', '</s>']
['<s>', 'I', 'read', 'a', 'different', 'book', '</s>']
['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']

Target sentence:
['<s>', 'I', 'read', 'a', 'different', 'book', 'by', 'Danielle', '</s>']


## 2. Counts

In [4]:
def bigrams(sentence: list[str]) -> list[tuple[str, str]]:
    return list(zip(sentence, sentence[1:]))


unigram_counts = Counter(token for sentence in training_sentences for token in sentence)
bigram_counts = Counter(pair for sentence in training_sentences for pair in bigrams(sentence))
vocabulary = sorted(unigram_counts)
vocabulary_size = len(vocabulary)

print('Vocabulary size:', vocabulary_size)
print(vocabulary)

Vocabulary size: 10
['</s>', '<s>', 'Danielle', 'He', 'I', 'a', 'book', 'by', 'different', 'read']


## 3. Unsmoothed Bigram Probability

In [5]:
@dataclass(frozen=True)
class BigramStep:
    bigram: tuple[str, str]
    numerator: int
    denominator: int
    probability: Fraction


def multiply(probabilities: list[Fraction]) -> Fraction:
    result = Fraction(1, 1)
    for probability in probabilities:
        result *= probability
    return result


unsmoothed_steps = []
for previous_token, token in bigrams(target_sentence):
    count = bigram_counts[(previous_token, token)]
    context_count = unigram_counts[previous_token]
    unsmoothed_steps.append(
        BigramStep(
            bigram=(previous_token, token),
            numerator=count,
            denominator=context_count,
            probability=Fraction(count, context_count),
        )
    )

unsmoothed_probability = multiply([step.probability for step in unsmoothed_steps])

unsmoothed_table = pd.DataFrame(
    {
        'Bigram': [f'{a} {b}' for a, b in [step.bigram for step in unsmoothed_steps]],
        'Count': [step.numerator for step in unsmoothed_steps],
        'Context Count': [step.denominator for step in unsmoothed_steps],
        'Probability': [f'{step.probability.numerator}/{step.probability.denominator}' for step in unsmoothed_steps],
    }
)

print('Unsmoothed sentence probability:', unsmoothed_probability, float(unsmoothed_probability))
unsmoothed_table

Unsmoothed sentence probability: 1/27 0.037037037037037035


,Bigram,Count,Context Count,Probability
0,<s> I,1,3,1/3
1,I read,1,1,1/1
2,read a,3,3,1/1
3,a different,1,3,1/3
4,different book,1,1,1/1
5,book by,1,3,1/3
6,by Danielle,1,1,1/1
7,Danielle </s>,1,1,1/1


## 4. Add-One Smoothed Bigram Probability

In [6]:
smoothed_steps = []
for previous_token, token in bigrams(target_sentence):
    count = bigram_counts[(previous_token, token)] + 1
    context_count = unigram_counts[previous_token] + vocabulary_size
    smoothed_steps.append(
        BigramStep(
            bigram=(previous_token, token),
            numerator=count,
            denominator=context_count,
            probability=Fraction(count, context_count),
        )
    )

smoothed_probability = multiply([step.probability for step in smoothed_steps])

smoothed_table = pd.DataFrame(
    {
        'Bigram': [f'{a} {b}' for a, b in [step.bigram for step in smoothed_steps]],
        'Smoothed Count': [step.numerator for step in smoothed_steps],
        'Smoothed Context Count': [step.denominator for step in smoothed_steps],
        'Probability': [f'{step.probability.numerator}/{step.probability.denominator}' for step in smoothed_steps],
    }
)

print('Smoothed sentence probability:', smoothed_probability, float(smoothed_probability))
smoothed_table

Smoothed sentence probability: 512/418161601 1.2244070205767171e-06


,Bigram,Smoothed Count,Smoothed Context Count,Probability
0,<s> I,2,13,2/13
1,I read,2,11,2/11
2,read a,4,13,4/13
3,a different,2,13,2/13
4,different book,2,11,2/11
5,book by,2,13,2/13
6,by Danielle,2,11,2/11
7,Danielle </s>,2,11,2/11
